In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
df = pd.read_csv('rps_data.csv', sep=',')

In [2]:
df['opp_match_wins'] = pd.to_numeric(df['opp_match_wins'], errors='coerce').fillna(-1).astype(int)
df['opp_match_winrate'] = pd.to_numeric(df['opp_match_winrate'], errors='coerce').fillna(-1.0)
df['score_diff'] = df['score_me_before'] - df['score_opp_before']
df['prev2_opp_move'] = df.groupby('match_id')['opp_move'].shift(2)
df['prev2_opp_move'] = df['prev2_opp_move'].fillna('-1')
df['next_opp_move'] = df.groupby('match_id')['opp_move'].shift(-1)
df = df.dropna(subset=['next_opp_move']).copy()

In [3]:
df.info()

<class 'pandas.DataFrame'>
Index: 137 entries, 0 to 161
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   match_id           137 non-null    int64  
 1   round              137 non-null    int64  
 2   opp_match_wins     137 non-null    int64  
 3   opp_match_winrate  137 non-null    float64
 4   stake              137 non-null    int64  
 5   opp_move           137 non-null    str    
 6   my_move            137 non-null    str    
 7   outcome            137 non-null    str    
 8   score_me_before    137 non-null    int64  
 9   score_opp_before   137 non-null    int64  
 10  prev_opp_move      137 non-null    str    
 11  prev_outcome       137 non-null    str    
 12  streak_draws       137 non-null    int64  
 13  prev2_opp_move     137 non-null    str    
 14  score_diff         137 non-null    int64  
 15  next_opp_move      137 non-null    str    
dtypes: float64(1), int64(8), str(7)
memory usa

In [4]:
df

,match_id,round,opp_match_wins,opp_match_winrate,stake,opp_move,my_move,outcome,score_me_before,score_opp_before,prev_opp_move,prev_outcome,streak_draws,prev2_opp_move,score_diff,next_opp_move
0,1,1,-1,-1.00,25,Н,Б,lose,0,0,-1,none,0,-1,0,К
1,1,2,-1,-1.00,25,К,Н,lose,0,1,Н,lose,0,-1,-1,Н
2,1,3,-1,-1.00,25,Н,Н,draw,0,2,К,lose,0,Н,-2,Н
3,1,4,-1,-1.00,25,Н,Н,draw,0,2,Н,draw,1,К,-2,Н
4,1,5,-1,-1.00,25,Н,К,win,0,2,Н,draw,2,Н,-2,Н
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156,25,5,24,54.55,25,Б,Б,draw,2,0,Н,win,0,К,2,К
158,26,1,0,0.00,50,К,К,draw,0,0,-1,none,0,-1,0,К
159,26,2,0,0.00,50,К,К,draw,0,0,К,draw,1,-1,0,К
160,26,3,0,0.00,50,К,Б,win,0,0,К,draw,2,К,0,К


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

le_target = LabelEncoder()
df['next_opp_move_enc'] = le_target.fit_transform(df['next_opp_move'])

# Определяем признаки (X) и целевую (y)
categorical_features = ['opp_move', 'my_move', 'outcome', 'prev_opp_move', 'prev_outcome', 'prev2_opp_move']
numeric_features = ['opp_match_wins', 'opp_match_winrate', 'stake', 
                    'score_me_before', 'score_opp_before', 'streak_draws']

X = df[categorical_features + numeric_features]
y = df['next_opp_move_enc']

# Разделяем на обучение и тест (стратификация по y)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# Препроцессор: OneHot для категорий, StandardScaler для чисел (хотя Random Forest не требует, но оставим)
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features),
    ('num', 'passthrough', numeric_features)   # без масштабирования, можно добавить StandardScaler()
])

# Модель Random Forest
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    random_state=42,
    class_weight='balanced'
)

# Пайплайн
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', rf)
])

In [7]:
# Обучение
pipeline.fit(X_train, y_train)

# Предсказание на тесте
y_pred = pipeline.predict(X_test)

# Оценка
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc:.3f}')
print(classification_report(y_test, y_pred, target_names=le_target.classes_))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.357
              precision    recall  f1-score   support

           Б       0.20      0.12      0.15         8
           К       0.50      0.73      0.59        11
           Н       0.14      0.11      0.12         9

    accuracy                           0.36        28
   macro avg       0.28      0.32      0.29        28
weighted avg       0.30      0.36      0.32        28

Confusion matrix:
[[1 2 5]
 [2 8 1]
 [2 6 1]]


In [8]:
# joblib.dump(pipeline, 'rps_model.pkl')
# joblib.dump(le_target, 'target_encoder.pkl')